# NN1 P_base — 계획기 없는 신경망 모델 재기 (무리 도착 세계)

계획서 1단계입니다. 신경망 **N1·N2·N4** 를 학습 6일로 만들어 같은 84상황에서 잽니다. 실행 때 계획기를 부르면 멈추고, 제어 60분과 꼬리 15분을 모두 신경망이 맡습니다.
방법과 읽는 법은 `results/explore/nn1_20261102/사전등록.md`에 고정했습니다. 이 노트북의 값은 흉내와 첫 학생까지이고 강화학습 라운드는 아직 아닙니다.

노트북은 넷(P_base·P_aug·R_base·R_aug)과 판정 하나입니다. **넷을 런타임 하나씩 따로** 씁니다. 순서는 상관없고 동시에 돌려도 됩니다.

1. 런타임 유형 **CPU + 높은 RAM(CPU 8개)** 을 고르고 **런타임 → 모두 실행**을 누르세요. GPU는 필요 없습니다. Drive 접근을 허용하세요.
2. 순서: 묶음 검증 → 패키지 → 시험 → 모델 학습(약 2분) → 경로 점검(3판, 약 5분) → 본 평가(756판, CPU 8개 기준 약 2.5시간, 어림).
3. 본 평가는 백그라운드로 돕니다. 끊기면 다시 **모두 실행**하세요. 끝난 판은 건너뜁니다.
4. 넷이 끝나면 `NN1_judge`를 실행하고 "끝났어"라고 알려 주세요. 결과: Drive `FutureTraffic/explore_nn1/`.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json

drive.mount("/content/drive")
WORLD, PART = "P", "base"
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_NN1.zip"
DATA = DRIVE / "explore_platoon1/data"                                # 기존 자료(읽기만)
TEACHER = DRIVE / "explore_platoon1/students/seed_42"                          # 선생님 모델(읽기만)
EVALUATION = DRIVE / "explore_platoon0"                    # 평가 지문 대조(읽기만)
PLATOON1C = next((p for p in (DRIVE / "explore_platoon1c", DRIVE / "explore_platoon1c (1)") if (p / "eval" / "t20" / "models.json").exists()), None)
TEACHER_RUN = PLATOON1C / "eval" / "t20" / "runs" / "t20_m0"
OUT = DRIVE / "explore_nn1" / WORLD / PART
WORKSPACE = Path("/content/FutureTraffic_NN1")
REFERENCE = WORKSPACE / "results/explore/nn1_20261102/smoke_reference" / WORLD      # 맥에서 만든 점검용 모델
MIN_CPUS = 8
for path in (ARCHIVE, DATA / "experiment.json", TEACHER / "model.pt", EVALUATION / "experiment.json", TEACHER_RUN):
    assert path.exists(), f"필요한 파일이 없습니다: {path}"
CPUS = os.cpu_count() or 1
print("CPU 개수:", CPUS, "| 세계:", WORLD, "| 결과 폴더:", OUT)
assert CPUS >= MIN_CPUS, (f"CPU가 {CPUS}개입니다. 런타임 → 런타임 유형 변경에서 CPU와 '높은 RAM'을 고른 뒤 다시 실행하세요.")

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "NN1_V1", "NN1 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("nn1_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]} "
      f"| PLATOON0 설정 {package_manifest['platoon0_config_sha256'][:12]} | 측정 계획 {package_manifest['plan_sha256'][:12]}")

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
# ARENA3 심판 기준선과 같은 평가 지문이 되도록 같은 패키지 버전을 씁니다.
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(평가 지문이 달라져 기준선과 비교할 수 없어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", "test_explore_nn1.py"], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 1) 모델 학습(약 2분). 이미 있으면 다시 만들지 않습니다.
args = ["src/explore_nn1.py", "train", "--world", WORLD, "--data", str(DATA), "--out", str(OUT / "models")]
if WORLD == "P":
    args += ["--teacher", str(TEACHER)]
colab.run_live(args, WORKSPACE)
print("모델 학습 끝")

In [ ]:
# 2) 경로 점검(약 5분): 맥 참조 모델을 기존 경로와 계획기 없는 경로로 돌려 신호·입력이 같은지, 계획기 호출이 0인지, 맥과 값이 같은지 봅니다.
CHECK = OUT / "check_paths"
previous = colab.read_json(CHECK / "check_paths.json")
if previous and previous.get("ok"):
    print("이전 점검이 통과해 건너뜁니다.")
else:
    colab.run_live(["src/explore_nn1.py", "check-paths", "--world", WORLD, "--model", str(REFERENCE), "--out", str(CHECK), "--work", "/content/n1_paths",
                    "--workers", "3", "--situations", "1", "--evaluation", str(EVALUATION)], WORKSPACE, check=False)
r = colab.read_json(CHECK / "check_paths.json")
assert r and r.get("ok"), f"점검 실패 — 이 칸의 출력을 저에게 보내 주세요: {r and r.get('checks')}"
print("경로 점검 통과:", r["checks"], "| 판당 초", r["seconds"], "| 꼬리 신경망 − 꼬리 P0:", r["tail_nn_minus_p0_s"])

In [ ]:
# 3) 본 평가. 백그라운드로 돌아 셀을 멈춰도 계속 돕니다.
#    런타임이 끊기면 처음 네 칸 → 이 칸을 다시 실행하세요. 끝난 판은 건너뜁니다.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요. 런타임이 끊기면 변수와 작업 폴더가 사라집니다."
CANDIDATES = "N1,N2,N4"
TOTAL = len(CANDIDATES.split(",")) * 3 * 84
LOGS = OUT / "logs"

def describe(name):
    done = len(list((OUT / "eval").glob("*/runs/*/*/cache.json")))
    failed = len(list((OUT / "eval").glob("*/runs/*/*.failed.json")))
    return f"끝난 판 {done}/{TOTAL}, 실패 기록 {failed}"

pids = colab.launch({f"nn1_{WORLD}_base": ["src/explore_nn1.py", "evaluate", "--world", WORLD, "--candidates", CANDIDATES, "--models", str(OUT / "models"),
                                            "--out", str(OUT / "eval"), "--work", "/content/n1", "--workers", str(CPUS)]}, WORKSPACE, LOGS)
code = colab.watch(pids, LOGS, 300, describe)
assert code == 0, "본 평가에서 오류가 났습니다. 위 로그를 저에게 보내 주세요(다시 실행하면 끝난 판은 건너뜁니다)."
print("본 평가 끝. 노트북 넷이 끝나면 NN1_judge를 실행하세요.")